### Installiamo `gurobipy`, documentazione @ https://www.gurobi.com/documentation/9.5/refman/py_python_api_details.html#sec:Python-details

In [ ]:
!pip3 install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 76.7 MB/s eta 0:00:00


# Importiamo i pacchetti necessari

In [2]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Volgiamo definire e risolvere il seguente problema

\begin{align}
  \text{maximize}\ &250x_1 + 230x_2 + 110x_3 + 350x_4 + 110x_5\\
                  &s.t.\\
                  &0\le\ x_i \lt \infty\\
                  &2x_1 + 1.5x_2 + 0.5x_3 + 2.5x_4 + 0.7x_5\leq 100.\\
                  &0.5x_1 + 0.25x_2 + 0.25x_3 + x_4 + 0.3x_5\leq 50\\
                  &x_1 + x_2 \geq 10\\
                  &x_3 = x_5\\
\end{align}

Creiamo il modello

In [3]:
model = gp.Model()

Restricted license - for non-production use only - expires 2027-11-29


Aggiungiamo le variabili con i rispettivi limiti

In [4]:
n_vars = 5
#addMVar mi fa aggiungere variabili come una lista con lower bound e upper bound (in questo caso 0 e infinito)
x = model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
model.update()

In [5]:
print(f"Model vars:{model.getVars()}\nNum vars: {len(model.getVars())}\nVar C0: {model.getVarByName('C0')}")

Model vars:[<gurobi.Var C0>, <gurobi.Var C1>, <gurobi.Var C2>, <gurobi.Var C3>, <gurobi.Var C4>]
Num vars: 5
Var C0: <gurobi.Var C0>


Aggiungiamo i vincoli del problema

In [ ]:


#A sono i coeff. nei vincoli (in orizzontale)
b = np.array([100, 50, 10, 0])
#b coeff. dei termini noti
ct = model.addConstr(A[:2]@x <= b[:2])
ct2 = model.addConstr(A[-2]@x >= b[-2])
ct3 = model.addConstr(A[-1]@x == b[-1])

model.update()


NameError: name 'model' is not defined

Definiamo la funzione obiettivo

In [8]:
obj_coefs = np.array([250, 230, 110, 350, 110])
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)

Risolviamo il problema

In [9]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model fingerprint: 0x98c1760a
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.04s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8333333e+04   2.500000e+00   0.000000e+00      0s
       1    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.08 seconds (0.00 work units)
Optimal objective  1.788333333e+04


Stampiamo per ogni variabile il valore nella soluzione ottima



In [10]:
for v in model.getVars():
  #attributo X (maiuscolo) è il valore della variabile ottima
  print('%s %g' % (v.VarName, v.X))

print('Obj: %g' % model.ObjVal)

C0 0
C1 10
C2 70.8333
C3 0
C4 70.8333
Obj: 17883.3


# Proviamo a risolvere il problema duale

In [7]:
dual_model = gp.Model()

In [8]:
n_vars = 4
#addMVar mi fa aggiungere variabili come una lista con lower bound e upper bound (in questo caso 0 e infinito)
x = dual_model.addMVar(n_vars, lb=[0, 0, -GRB.INFINITY, -GRB.INFINITY], ub=[GRB.INFINITY, GRB.INFINITY, 0, GRB.INFINITY])
dual_model.update()

In [9]:
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])
newA = A.T
#A sono i coeff. nei vincoli (in orizzontale)
c = np.array([250, 230, 110, 350, 110])
#b coeff. dei termini noti
ctduale = dual_model.addConstr(newA@x >= c)

dual_model.update()

In [10]:
#nuova f obiettivo
b = np.array([100, 50, 10, 0])
dual_model.setObjective(b @ x, GRB.MINIMIZE)

In [11]:
dual_model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 5 rows, 4 columns and 14 nonzeros (Min)
Model fingerprint: 0x0d6becd1
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 4e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.03s
Presolved: 4 rows, 3 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.0000000e+31   1.500000e+30   1.000000e+01      0s
       3    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.04 seconds (0.00 work units)
Optimal objective  1.788333333e+04


In [1]:
for v in dual_model.getVars():
  #attributo X (maiuscolo) è il valore della variabile ottima
  print('%s %g' % (v.VarName, v.X))

print('Obj: %g' % dual_model.ObjVal)

NameError: name 'dual_model' is not defined

# E' possibile accedere ai vincoli ed alla funzione obiettivo

In [ ]:
for c in dual_model.getConstrs():
  print(dual_model.getRow(c))

In [ ]:
dual_model.getObjective()

# Rendiamo il problema primale privo di soluzioni ammissibili

In [5]:
#basta aggiungere un vincolo che non potrà venir soddisfatto
dual_model1 = gp.Model()
n_vars = 4
#addMVar mi fa aggiungere variabili come una lista con lower bound e upper bound (in questo caso 0 e infinito)
x = dual_model1.addMVar(n_vars, lb=[0, 0, -GRB.INFINITY, -GRB.INFINITY], ub=[GRB.INFINITY, GRB.INFINITY, 0, GRB.INFINITY])
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])
newA = A.T
#A sono i coeff. nei vincoli (in orizzontale)
c = np.array([250, 230, 110, 350, 110])
#b coeff. dei termini noti
ctduale = dual_model1.addConstr(newA@x >= c)
dual_model1.update()
b = np.array([100, 50, 10000, 0])#lo rendo impossibile
dual_model1.setObjective(b @ x, GRB.MINIMIZE)
dual_model1.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 5 rows, 4 columns and 14 nonzeros (Min)
Model fingerprint: 0x90612b54
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [5e+01, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 4e+02]

Presolve time: 0.02s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Infeasible or unbounded model


In [ ]:
status_dict = {GRB.OPTIMAL: "OPTIMAL"}

In [ ]:
print(dual_model1.getAttr('Status'))

4


: 

In [5]:
#SE VOGLIO CHE ABBIA SOLUZIONE MA NON CE L HA USO QUESTO
for c in dual_model1.getConstrs():
    print(c.getAttr('IISConstr'))

AttributeError: Unable to retrieve attribute 'IISConstr'

# Rendiamo il problema primale illimitato

In [ ]:
#basta aggiungere un vincolo che non potrà venir soddisfatto
dual_model2 = gp.Model()
dual_model2.setParam(GRB.Param.DualReductions, 0)
n_vars = 4
#addMVar mi fa aggiungere variabili come una lista con lower bound e upper bound (in questo caso 0 e infinito)
x = dual_model2.addMVar(n_vars, lb=[0, 0, -GRB.INFINITY, -GRB.INFINITY], ub=[GRB.INFINITY, GRB.INFINITY, 0, GRB.INFINITY])
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])
newA = A.T
#A sono i coeff. nei vincoli (in orizzontale)
c = np.array([250, 230, 110, 350, 110])
#b coeff. dei termini noti
ctduale = dual_model2.addConstr(newA@x >= c)
dual_model2.update()
b = np.array([100, 50, 10, 0])#lo rendo impossibile
dual_model2.setObjective(b @ x, GRB.MAXIMIZE)
dual_model2.optimize()

# Problema del cammino minimo
Dato un grafo $\mathcal{G}$ e due nodi $s,\ t$ vogliamo trovare il cammino che minimiza il costo di andare da $s$ a $t$.

Creiamo un grafo costituito da $200$ nodi

In [3]:
n_nodes = 200
adj_mat = np.random.rand(n_nodes, n_nodes)
adj_mat[adj_mat > 0.02] = 0
adj_mat *= 1000
adj_mat = np.round(adj_mat)
np.fill_diagonal(adj_mat, 0)

Definiamo il nodo di partenza ed il nodo di arrivo

In [4]:
s = 0
t = 197

## Risolviamo il problema con Gurobi

Se il costo di ogni arco e' positivo, il problema si puo' formulare nel seguente modo:
\begin{align}
  \text{minimize} &\sum_{ij} c_{ij}x_{ij}\\
                  & s.t.\\
                  0 \leq\ &x \leq 1\\
                  \sum_i x_{iv} + b_v &= \sum_j x_{vj},\ \forall v \in \text{Nodes}
\end{align}
dove $b_v$ il flusso in ingresso su ogni nodo ed $x_{ij}$ la quantita' di flusso nell'arco $(i, j)$.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

Creiamo il modello

In [ ]:
m = gp.Model('shortestPath')

Aggiungiamo le variabili e definiamo la funzione obiettivo

In [ ]:
var_idxs = np.nonzero(adj_mat)
#mi da gli indici di dove ho archi (visto dalla matrice di adiacenza)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]
costs = {idx: adj_mat[idx] for idx in var_idxs}

#addvars crea variabili per ogni tupla nella lista che gli passo
#mettendo anche obj gli do già i coeff. per la funzione obiettivo
#lui la fa in automatico sapendo che la funzione è somma di coeff * var
arcs = m.addVars(var_idxs, lb = 0, ub = 1, obj=costs, name="arcs")
m.update()

In [ ]:
m.getObjective()

Aggiungiamo i vincoli

In [5]:
b = np.zeros(n_nodes)
b[s] = 1
b[t] = -1

#arcs.sum è una sommatoria
#NB comodo perché è proprio uguale alla scrittura matematica
ct = m.addConstrs((arcs.sum('*', v) + b[v] == arcs.sum(v, '*') for v in range(n_nodes)))

NameError: name 'm' is not defined

Risolviamo il problema e stampiamo la soluzione trovata

In [ ]:
m.optimize()

In [ ]:
print(f"Shortest path cost: {m.objVal:.4f}")
for v in var_idxs:
  if arcs[v].getAttr('X') == 1:
    print(f"{v[0]:3d} -> {v[1]:3d}: {arcs[v].getAttr('Obj'):.4f}")

# Misc Utili

In [6]:
a = [1,2,3,4]
b = ["a", "b", "c", "d"]

## Itertools
Implementa molte funzioni utili per iterare su collezioni di elementi

In [7]:
import itertools

In [8]:
print(list(itertools.chain(a, b)))

[1, 2, 3, 4, 'a', 'b', 'c', 'd']


In [ ]:
print(list(itertools.product(a,b)))

[(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'a'), (2, 'b'), (2, 'c'), (2, 'd'), (3, 'a'), (3, 'b'), (3, 'c'), (3, 'd'), (4, 'a'), (4, 'b'), (4, 'c'), (4, 'd')]


## zip, enumerate

In [ ]:
for a_i, b_i in zip(a, b):
  print(a_i, b_i)

1 a
2 b
3 c
4 d


In [ ]:
for x in zip(a, b):
  print(x)

(1, 'a')
(2, 'b')
(3, 'c')
(4, 'd')


In [ ]:
for i, a_i in enumerate(a):
  print(i, a_i)

0 1
1 2
2 3
3 4


In [ ]:
for x in enumerate(a):
  print(x)

(0, 1)
(1, 2)
(2, 3)
(3, 4)


# Esercizio

Un polo industriale produce televisori utilizzando tre linee di montaggio (A, B e C) per televisori piatti, curvi e monitor ad alte prestazioni. Per stabilire la produzione per il prossimo semestre e’ necessario considerare I seguenti aspetti:

  * Dalla vendita dei tre prodotti l’azienda ricava un profitto pari a 200, 400 e 500 euro rispettivamente.

  * Nel corso dei sei mesi, la linea A potra’ essere impengata per un totale di 150 ore al mese, la linea B per 130 al mese e la linea C per 170 ore al mese. Tuttavia, esclusivamente per il quarto mese di produzione sarà possibile usufruire di ulteriori 20 ore di lavoro sulla linea A e di 10 ore in meno sulla linea C.

  * Produrre un televisore a schermo piatto richiede 18 ore sulla linea A e 7 sulla B, un televisore curvo invece ne richiede 12 sulla linea B e 25 sulla C, mentre un monitor ad alte prestazioni richiede 10 ore di lavorazione sulla linea A, 9 sulla linea B e 14 sulla linea C.

  * In un impianto separato l’azienda esegue verifiche sui prodotti ed ha una capacita’ lavorativa mensile pari a 120 ore. Per eseguire tutte le verifiche necessarie sono richieste 5 ore per un televisore a schermo piatto, 8 per uno curvo e 9 per un monitor.

  * A seguito di analisi di mercato vi sono I seguenti vincoli sulla produzione: nel secondo mese la produzione di monitor deve supereare le 10 unita’, nel terzo e quinto mese la produzione di televisori curvi deve essere almeno pari al 30% di quella di televisori piatti ed infine nel sesto mese di produzione non e’ prevista richiesta di monitor ad alte prestazioni.

  * Temendo uno sciopero dei lavoratori, la direzione considera che nel primo mese di produzione la capacita’ lavorativa dell’impianto di verifica dei prodotti sara’ ridotta al 75%.

  * La produzione complessiva sui 6 mesi di monitor ad alte prestazioni non deve superare le 40 unità.

Sulla base di tali considerazioni, il problema e’ pianificiare la produzione del prossimo semestre con l’obiettivo di massimizzare il profitto dell’azienda.

In [7]:
import gurobipy as gp
from gurobipy import GRB
import itertools

In [8]:
model = gp.Model()

Restricted license - for non-production use only - expires 2027-11-29


In [10]:
televisori = ['piatto', 'curvo', 'monitor']
mesi = range(6)
#mi creo degli array per tenere il termine noto dei vincoli di ogni mese
notiA = [150]*6 #lista con 6 volte 150
notiB = [130] * 6
notiC = [170] * 6
notiVerifica = [120] * 6

#quarto mese 20 ore in piu sulla A e 10 in meno sulla C
notiA[3] += 20
notiC[3] -= 10

#primo mese 75% delle ore nella verifica
notiVerifica[0] *= 0.75

tempi_piatti = [18, 7, 0, 5]#il 4 elemento è per la verifica
tempi_curvi = [0, 12, 25, 8]
tempi_monitor = [10, 9, 14, 9]

#x_P[t] = numero di tv piatti prodotti nel mese t
#x_C[t] = numero di tv curvi prodotti nel mese t
#x_M[t] = numero di monitor prodotti nel mese t
x_P = model.addVars(mesi, vtype=GRB.INTEGER, name="Piatti")
x_C = model.addVars(mesi, vtype=GRB.INTEGER, name="Curvi")
x_M = model.addVars(mesi, vtype=GRB.INTEGER, name="Monitor")
#quindi ho 6 variabili per ogni tipo, ognuna è un tipo in uno specifico mese

model.setObjective(
    gp.quicksum(200 * x_P[t] + 
                400 * x_C[t] + 
                500 * x_M[t] for t in mesi), 
    GRB.MAXIMIZE
)#sarebbe la somma di questa formula in tutti i mesi

for t in mesi:
    model.addConstr(tempi_piatti[0] * x_P[t] + tempi_curvi[0] * x_C[t] + tempi_monitor[0] * x_M[t] <= notiA[t])
    
    model.addConstr(tempi_piatti[1] * x_P[t] + tempi_curvi[1] * x_C[t] + tempi_monitor[1] * x_M[t] <= notiB[t])
    
    model.addConstr(tempi_piatti[2] * x_P[t] + tempi_curvi[2] * x_C[t] + tempi_monitor[2] * x_M[t] <= notiC[t])
    
    model.addConstr(tempi_piatti[3] * x_P[t] + tempi_curvi[3] * x_C[t] + tempi_monitor[3] * x_M[t] <= notiVerifica[t])

#secondo mese: monitor > 10
model.addConstr(x_M[1] >= 11)

#terzo mese e quinto mese: curvi >= 30% dei piatti
model.addConstr(x_C[2] >= 0.3 * x_P[2])
model.addConstr(x_C[4] >= 0.3 * x_P[4])

#sesto mese: 0 monitor
model.addConstr(x_M[5] == 0)

model.optimize()

if model.status == GRB.OPTIMAL:
    for t in mesi:
        print(f"Mese {t+1}:")
        print(f"  TV Piatti:  {int(x_P[t].X)}")
        print(f"  TV Curvi:   {int(x_C[t].X)}")
        print(f"  Monitor:    {int(x_M[t].X)}")
else:
    print("Il modello non ha trovato una soluzione ottimale")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 48 rows, 36 columns and 116 nonzeros (Max)
Model fingerprint: 0x143653f0
Model has 18 linear objective coefficients
Variable types: 0 continuous, 36 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-01, 2e+01]
  Objective range  [2e+02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 2e+02]

Found heuristic solution: objective 23500.000000


Presolve removed 43 rows and 33 columns
Presolve time: 0.04s
Presolved: 5 rows, 3 columns, 12 nonzeros
Found heuristic solution: objective 27100.000000
Variable types: 0 continuous, 3 integer (0 binary)
Found heuristic solution: objective 29900.000000

Root relaxation: objective 3.361225e+04, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 33612.2530    0    3 29900.0000 33612.2530  12.4%     -    0s
H    0     0                    32900.000000 33612.2530  2.16%     -    0s
     0     0 33531.7073    0    3 32900.0000 33531.7073  1.92%     -    0s
H    0     0                    33300.000000 33531.7073  0.70%     -    0s
     0     0 33531.7073    0    3 33300.0000 33531.7073  0.70%     -    0s

Cutting planes:
  Gomory: 1

Explored 1 nodes (3 simplex iterations) in 0.15 seconds (0.00 work units)
Thread count was 4 (of 4 avail